<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/02_digital_twin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 2 of 7: From 3D Image to Device Model

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

A common workflow in electrochemical research is to characterise a 3D microstructure from X-ray tomography and feed the resulting transport parameters into a continuum device model such as [PyBaMM](https://pybamm.org/). This tutorial walks through that pipeline end-to-end.

**What you will learn:**
1. Load a real 3D TIFF image stack.
2. Compute directional tortuosity (X, Y, Z) to quantify anisotropy.
3. Visualise the 3D diffusion field using the C++ core API and `yt`.
4. Export results to the BPX (Battery Parameter eXchange) JSON format.
5. Import those parameters into PyBaMM and run a discharge simulation.

**Prerequisites:** [Tutorial 1](01_hello_openimpala.ipynb) — basic OpenImpala workflow (volume fraction, percolation, tortuosity).

In [ ]:
# Install OpenImpala (compiled C++ backend needed for low-level API in this tutorial)
!pip install -q openimpala-cuda --find-links https://github.com/BASE-Laboratory/OpenImpala/releases/latest nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 nvidia-cusparse-cu12 nvidia-curand-cu12 pybamm bpx tifffile matplotlib yt

In [ ]:
import os
import time
import json
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tifffile

import openimpala as oi
from openimpala import core as oic  # Low-level C++ API (pybind11 wrappers)
import pybamm

print(f"OpenImpala version {oi.__version__} loaded.")

# In a notebook we open the session manually so it stays alive across
# cells.  In a script, use `with oi.Session():` instead (see Tutorial 1).
session = oi.Session()
session.__enter__()

# Download the sample TIFF from the OpenImpala repository so that the
# notebook works in ephemeral environments like Google Colab.
data_url = "https://raw.githubusercontent.com/BASE-Laboratory/OpenImpala/master/data/SampleData_2Phase_stack_3d_1bit.tif"
data_path = "SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    print("Downloading sample data...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete.")

## 1. Loading Tomography Data

OpenImpala's C++ backend includes readers for TIFF, RAW, and HDF5. When using the Python API, the most flexible approach is to load your data with a standard library (e.g. `tifffile`, `h5py`), apply any preprocessing you need (cropping, filtering, relabelling), and then hand the resulting NumPy array to OpenImpala.

In [ ]:
# Read the TIFF stack into a 3D NumPy array (Z, Y, X)
microstructure = tifffile.imread(data_path).astype(np.int32)

print(f"Loaded image shape (Z, Y, X): {microstructure.shape} ({microstructure.size / 1e6:.1f}M voxels)")
print(f"Unique phases present:        {np.unique(microstructure)}")

# Visualize the middle slice
z_mid = microstructure.shape[0] // 2
plt.figure(figsize=(5, 5), dpi=100)
plt.imshow(microstructure[z_mid, :, :], cmap='gray')
plt.title(f"Microstructure Slice at Z = {z_mid}", fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Directional Tortuosity and Anisotropy

We treat **Phase 1 as the pore space** (electrolyte). Manufactured electrodes are often calendered (compressed), which preferentially restricts transport in the through-thickness direction. Computing the tortuosity along all three cardinal directions reveals the extent of this anisotropy.

In [ ]:
directions = ['x', 'y', 'z']
tau_results = {}

print("Computing directional tortuosity...")
t_start_all = time.time()

# 1. Volume fraction
vf = oi.volume_fraction(microstructure, phase=1)
print(f"Pore volume fraction (porosity): {vf.fraction:.4f}\n")

# 2. Tortuosity for each direction
for d in directions:
    t0 = time.time()
    perc = oi.percolation_check(microstructure, phase=1, direction=d)

    if perc.percolates:
        res = oi.tortuosity(microstructure, phase=1, direction=d, solver="flexgmres", max_grid_size=128)
        tau_results[d.upper()] = res.tortuosity
        t_solve = time.time() - t0
        print(f"  {d.upper()}-direction: tau = {res.tortuosity:.4f} ({t_solve:.2f}s)")
    else:
        tau_results[d.upper()] = None
        print(f"  {d.upper()}-direction: does not percolate.")

t_total = time.time() - t_start_all
print(f"\nAll three directions solved in {t_total:.2f}s ({microstructure.size / 1e6:.1f}M voxels).")

The bar chart below compares the tortuosity across directions. A value closer to 1.0 indicates less resistance to transport. Differences between directions reflect structural anisotropy in the material.

In [ ]:
dirs = list(tau_results.keys())
taus = [v for v in tau_results.values() if v is not None]

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
bars = ax.bar(dirs, taus, color=['#1f77b4', '#ff7f0e', '#2ca02c'], width=0.5)
ax.axhline(y=1.0, color='r', linestyle='--', label='Ideal limit (Tau=1.0)')

ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontweight='bold')
ax.set_title("Directional Tortuosity (Anisotropy)", fontweight='bold')
ax.set_ylim(1.0, max(taus) * 1.15)
ax.legend()

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.02,
            f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

## 3. Visualising the 3D Diffusion Field

The high-level `oi.tortuosity()` function skips field output by default to save time and disk space. To inspect the solved potential field, we can use the `openimpala.core` API with `write_plotfile=True`. The resulting AMReX plotfile can then be visualised with [yt](https://yt-project.org/), which has native support for AMReX data.

In [ ]:
import glob
import yt

# Create a directory for AMReX plotfiles
out_dir = "./plotfiles"
os.makedirs(out_dir, exist_ok=True)

print("Running core solver to generate 3D plotfile...")

# Convert the NumPy array into an OpenImpala VoxelImage
img = oi.VoxelImage.from_numpy(microstructure.copy(), max_grid_size=128)

# Instantiate the C++ HYPRE solver with plotfile output enabled
solver = oic.TortuosityHypre(
    img,
    vf.fraction,               # Volume fraction (calculated earlier)
    1,                         # Phase ID
    oic.Direction.Z,           # Solve in Z direction
    oic.SolverType.FlexGMRES,  # HYPRE solver
    out_dir,                   # Output directory
    0.0, 1.0,                  # Boundary conditions (V_lo, V_hi)
    1,                         # Verbose level
    True                       # Write plotfile
)

# Execute the solve
solver.value()
print("Plotfile generation complete.")

In [ ]:
# Find the generated AMReX plotfile directory
plotfile_dirs = [d for d in glob.glob(f"{out_dir}/*") if os.path.isdir(d)]
latest_plotfile = sorted(plotfile_dirs)[-1]

print(f"Loading AMReX data from: {latest_plotfile}")

# Load the data into yt (suppress verbose logging)
yt.funcs.mylog.setLevel(40)
ds = yt.load(latest_plotfile)

# The C++ solver names the field 'solution_potential'
field_name = ('boxlib', 'solution_potential')

# Slice through the middle of the Y-axis
slc = yt.SlicePlot(ds, normal='y', fields=field_name)

# Use a linear scale (the potential ranges from 0 to 1)
slc.set_log(field_name, False)
slc.set_zlim(field_name, 0.0, 1.0)
slc.set_cmap(field_name, 'magma')
slc.annotate_title("Steady-State Diffusion Potential (Z-Direction)")

slc.show()

## 4. Exporting to BPX Format

OpenImpala includes a `ResultsJSON` builder that produces files compliant with the [Battery Parameter eXchange (BPX)](https://github.com/pybop-team/BPX) standard.

Recall that the effective diffusivity is defined as $D_\text{eff} = D_\text{bulk}\,\varepsilon / \tau$, so the transport efficiency (the ratio $D_\text{eff}/D_\text{bulk}$) is $\varepsilon/\tau$.

In [ ]:
# Access the low-level C++ bindings via openimpala.core
rj = oic.ResultsJSON()

# Define the physics we just solved (Diffusion)
physics_cfg = oic.PhysicsConfig.from_type_string("diffusion")
rj.set_physics_config(physics_cfg)

# Set metadata
rj.set_input_file(data_path)
rj.set_phase_id(1)
rj.set_volume_fraction(vf.fraction)

# Add our directional results
for d, tau in tau_results.items():
    if tau is not None:
        transport_efficiency = vf.fraction / tau
        rj.add_direction_result(d, transport_efficiency)

# Enable BPX fragment generation for the Positive Electrode
rj.set_bpx_electrode("positive")
rj.set_provenance(sample_id="SampleData_2Phase", provenance_uri="https://github.com/BASE-Laboratory/OpenImpala")

# Build the JSON string and save to disk
bpx_json_string = rj.build_json_string()
bpx_filepath = "openimpala_cathode.json"

with open(bpx_filepath, "w") as f:
    f.write(bpx_json_string)

print(f"Exported BPX data to {bpx_filepath}")

## 5. Running a Device Simulation in PyBaMM

With the BPX JSON file in hand, we can feed the microstructure-derived parameters into PyBaMM and simulate a discharge curve. This closes the loop from 3D imaging to device-level prediction.

In [ ]:
print("Loading PyBaMM DFN model...")
model = pybamm.lithium_ion.DFN()

# Start from the Chen 2020 parameter set as a baseline
param = pybamm.ParameterValues("Chen2020")

# Read the BPX JSON we just exported
with open(bpx_filepath, "r") as f:
    oi_data = json.load(f)

# Extract the parameters OpenImpala calculated
cathode_params = oi_data["bpx"]["Parameterisation"]["Positive electrode"]
oi_porosity = cathode_params["Porosity"]
oi_bruggeman = cathode_params["Bruggeman coefficient (electrolyte)"]

print(f"Overriding defaults with OpenImpala results:")
print(f"  Positive electrode porosity:      {oi_porosity:.4f}")
print(f"  Positive electrode Bruggeman exp: {oi_bruggeman:.4f}")

# Inject our measured parameters
param.update({
    "Positive electrode porosity": oi_porosity,
    "Positive electrode Bruggeman coefficient (electrolyte)": oi_bruggeman
}, check_already_exists=False)

print("\nRunning 1C discharge simulation...")
sim = pybamm.Simulation(model, parameter_values=param)
solution = sim.solve([0, 3600])

# Plot the discharge curve
t = solution["Time [s]"].entries
V = solution["Terminal voltage [V]"].entries

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.plot(t, V, lw=2.5, color="#d62728")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Terminal Voltage (V)")
ax.set_title("PyBaMM Discharge Curve with OpenImpala Parameters")
ax.grid(True, linestyle="--", alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Next Steps

This tutorial demonstrated the full pipeline from a 3D tomography image to a device-level simulation, with BPX as the interchange format.

**Continue the series:**
- [Tutorial 3: REV and Uncertainty](03_rev_and_uncertainty.ipynb) — How large does your sample need to be for statistically representative results?
- [Tutorial 4: Effective Diffusivity and Field Visualisation](04_multiphase_and_fields.ipynb) — Extend to the cell-problem solver and multi-phase composites.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Analyse full-resolution synchrotron datasets with MPI.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **PyBaMM:** M. Sulzer et al., *Python Battery Mathematical Modelling (PyBaMM)*, J. Open Research Software 9(1), 14 (2021). [doi:10.5334/jors.309](https://doi.org/10.5334/jors.309)
3. **BPX standard:** *Battery Parameter eXchange (BPX)*, an open standard for lithium-ion battery parameterisation. [GitHub](https://github.com/pybop-team/BPX)
4. **Electrode tortuosity & anisotropy:** M. Ebner et al., *Tortuosity anisotropy in lithium-ion battery electrodes*, Advanced Energy Materials 4(5), 1301278 (2014). [doi:10.1002/aenm.201301278](https://doi.org/10.1002/aenm.201301278)
5. **yt visualisation:** M. J. Turk et al., *yt: A multi-code analysis toolkit for astrophysical simulation data*, Astrophys. J. Suppl. 192, 9 (2011). [doi:10.1088/0067-0049/192/1/9](https://doi.org/10.1088/0067-0049/192/1/9)
6. **Digital twin concept for batteries:** A. A. Franco et al., *Boosting rechargeable batteries R&D by multiscale modeling: myth or reality?*, Chemical Reviews 119(7), 4569–4627 (2019). [doi:10.1021/acs.chemrev.8b00239](https://doi.org/10.1021/acs.chemrev.8b00239)